# 고급 RAG 기법 — Ensemble + MMR으로 검색 품질 향상

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- 기본 RAG의 한계(벡터 검색만 사용할 때의 문제)를 명확히 설명하기
- **EnsembleRetriever**로 BM25와 FAISS를 결합하는 하이브리드 검색 구현하기
- **MMR(Maximum Marginal Relevance)**로 다양성 있는 검색 결과 얻기

## 사전 지식

- 00-RAG-Basic.ipynb의 8단계 RAG 파이프라인
- 6-5 섹션의 EnsembleRetriever와 MMR 개념

---

```mermaid
flowchart TD
    Q[사용자 질문]:::input --> B[BM25<br/>키워드 검색]:::process
    Q --> VM[FAISS + MMR<br/>의미+다양성 검색]:::process
    B -- 가중치 0.4 --> E[Ensemble Retriever<br/>RRF 결합]:::process
    VM -- 가중치 0.6 --> E
    E --> P[Prompt 생성]:::process
    P --> L[LLM 답변]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## 기본 RAG의 한계

| 문제 | 예시 |
|------|------|
| 키워드 미매칭 | "AI"로 검색했는데 문서엔 "인공지능"만 있는 경우 |
| 중복 결과 | 유사한 내용의 문서가 반복적으로 검색됨 |
| 동의어 처리 | "LLM" = "대규모 언어 모델" 연결 불가 |

> **실무 팁**: Ensemble의 가중치 `[BM25, FAISS]`를 `[0.3, 0.7]`처럼 의미 검색에 더 높은 가중치를 주면 자연어 질문에 더 잘 응답해요.

## 기본 RAG의 한계

### 1. 벡터 검색만 사용할 때의 문제

- **의미는 비슷하지만 정확한 키워드가 없으면 검색 실패**
  - 예: "AI"를 검색했는데 문서에는 "인공지능"만 있는 경우
- **빈도가 낮은 중요 키워드 놓침**
  - 예: 고유명사나 기술 용어

### 2. 검색 결과의 중복

- 유사한 내용의 문서들이 반복적으로 검색됨
- 다양한 관점의 정보 부족

### 해결책

- **Ensemble Retriever**: Dense(벡터) + Sparse(키워드) 검색 결합
- **MMR**: 관련성과 다양성을 동시에 고려

## 환경 설정

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## 1. Ensemble Retriever

### 개념

Ensemble Retriever는 여러 검색 방식을 결합하여 더 나은 검색 결과를 얻습니다.

```
                  User Query
                      |
         +------------+------------+
         |                         |
    BM25 검색               벡터 검색
  (키워드 기반)           (의미 기반)
         |                         |
         +------------+------------+
                      |
              Ensemble Retriever
            (가중치로 결합)
                      |
                최종 검색 결과
```

### BM25 vs 벡터 검색

| 특징 | BM25 (Sparse) | 벡터 검색 (Dense) |
|------|---------------|------------------|
| 방식 | 키워드 매칭 | 의미 유사도 |
| 장점 | 정확한 용어 검색 | 동의어, 유사 개념 검색 |
| 단점 | 의미 파악 불가 | 정확한 키워드 놓칠 수 있음 |
| 예시 | "GPT-4" 정확히 검색 | "대형 언어 모델" ≈ "LLM" |

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

### 문서 준비

In [ ]:
# 문서 로드 및 분할
FILE_PATH = "./data/2026_gov.pdf"
loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

split_documents = text_splitter.split_documents(docs[41:95])

len(split_documents)


56

In [9]:
# ---------------------------------------------------
# BM25 + FAISS Ensemble Retriever 구성
# ---------------------------------------------------
# ============================================================
# TODO: BM25Retriever와 FAISS Retriever를 만들고 EnsembleRetriever로 결합하세요
# 힌트: EnsembleRetriever(retrievers=[bm25, faiss], weights=[0.5, 0.5])
# 예상 결과: "Ensemble Retriever 생성 완료" 출력
# ============================================================

# 1단계: BM25 Retriever — 형태소 분석 없이 공백 기반 키워드 매칭
# ⚠️ 주의: 한국어는 형태소 분석 없이 BM25 정확도가 낮을 수 있음 — KiwiPirata 등 결합 고려
bm25_retriever = BM25Retriever.from_documents(split_documents)
bm25_retriever.k = 3

# 2단계: FAISS Retriever — 의미 기반 벡터 유사도 검색
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", chunk_size=300)
vectorstore = FAISS.from_documents(split_documents, embeddings)
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 3단계: Ensemble Retriever — RRF(Reciprocal Rank Fusion) 알고리즘으로 결합
# 💡 팁: weights 합계가 1.0일 필요는 없음 — RRF 알고리즘이 순위 기반으로 융합
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.5, .5]
)




In [13]:
# ---------------------------------------------------
# BM25 + FAISS Ensemble Retriever 구성
# ---------------------------------------------------
# ============================================================
# TODO: BM25Retriever와 FAISS Retriever를 만들고 EnsembleRetriever로 결합하세요
# 힌트: EnsembleRetriever(retrievers=[bm25, faiss], weights=[0.5, 0.5])
# 예상 결과: "Ensemble Retriever 생성 완료" 출력
# ============================================================

test_query = "장애인 구직자를 위한 경제적 지원 제도가 2026년에 어떻게 바뀌었나요?"

# 1단계: BM25 Retriever — 형태소 분석 없이 공백 기반 키워드 매칭
res = bm25_retriever.invoke(test_query)
for i, d in enumerate(res):
    d.page_content
    print(f' ==> [Line 15]: \033[38;2;213;236;47m[d.page_content]\033[0m({type(d.page_content).__name__}) = \033[38;2;9;94;230m{d.page_content}\033[0m')
# 2단계: FAISS Retriever — 의미 기반 벡터 유사도 검색
res = faiss_retriever.invoke(test_query)
for i, d in enumerate(res):
    d.page_content
    print(f' ==> [Line 20]: \033[38;2;129;61;80m[d.page_content]\033[0m({type(d.page_content).__name__}) = \033[38;2;126;138;139m{d.page_content}\033[0m')

# 3단계: Ensemble Retriever — RRF(Reciprocal Rank Fusion) 알고리즘으로 결합
res = ensemble_retriever.invoke(test_query)
for i, d in enumerate(res):
    d.page_content
    print(f' ==> [Line 26]: \033[38;2;44;194;179m[d.page_content]\033[0m({type(d.page_content).__name__}) = \033[38;2;78;120;87m{d.page_content}\033[0m')


 ==> [Line 15]: [d.page_content](str) = 24
장애인 표준사업장에 대한 세액감면 확대
재정경제부
www.moef.go.kr
장애인 일자리 창출 지원을 위하여, 장애인 표준사업장에 대한 소득세·법인세 감면율을 
확대합니다.
‌ 	 (종전) 3년 100% + 2년 50%
‌ 	 (개정) 3년 100% + 2년 50% + 5년 30%
개정내용은 2026년 1월 1일 이후 장애인 표준사업장으로 인증받은 내국인부터 적용됩니다.
장애인 표준사업장에 대한 세액감면 확대
추진배경
장애인 일자리 창출 지원
주요내용
장애인 표준사업장에 대한 소득세·법인세 감면율 확대 
시행일
2026년 1월 1일 이후 장애인 표준사업장으로 인증받은 내국인부터 적용
재정경제부 누리집>보도자료>2025년 세법개정안 기재위 전체회의 통과(2025. 11.)
재정경제부 조세특례제도과
☎ 044-215-4133
 ==> [Line 15]: [d.page_content](str) = 11
2026년부터 이렇게 달라집니다
초등 저학년 예체능 학원비 세제지원 
재정경제부
www.moef.go.kr
현재 시행 중인 교육비 세액공제(15%) 대상에 2026년부터 초등학교 저학년 자녀의 예체능 
학원비도 포함됩니다.
‌ 	 (초등학교 ‘저학년’ 기준) 과세기간 종료일 현재 만 9세 미만 공제
교육비 세액공제 주요대상
구분
주요대상
한도
본인 
· 대학원비  
· 직업능력개발훈련을 위한 수강료 
한도 없음
장애인
· 장애인재활교육을 위한 특수교육비 
취학전 아동
· 유치원·어린이집 수업료  
· 학원 및 체육시설업자에게 지급한 교육비 
300만원
초중고생 
대학생
· 수업료·입학금 등 공납금 
· 방과후학교 수업료 및 교재비(초중고생) 
· 급식비ㆍ교과서대금(초중고생) 
· 교복구입비(중고생, 한도 50만원) 
· 예체능 학원비(만 9세 미만)
초중고생 : 300만원 
대학생 : 900만원
개정내용은 2026년 1월 1일 이후 지출하는 분부터 적용합니다.
초등 저학년 예체능 학원

### 검색 비교 테스트

In [ ]:
# BM25만 사용
bm25_results = bm25_retriever.invoke(test_query)
print(f":mag: BM25 검색 결과 ({len(bm25_results)}개):")
print("=" * 60)
for i, doc in enumerate(bm25_results[:2], 1):
    print(f"[{i}] {doc.page_content[:150]}...")

# FAISS만 사용
faiss_results = faiss_retriever.invoke(test_query)
print(f"\n:mag: FAISS 검색 결과 ({len(faiss_results)}개):")
print("=" * 60)
for i, doc in enumerate(faiss_results[:2], 1):
    print(f"[{i}] {doc.page_content[:150]}...")

# Ensemble 사용
ensemble_results = ensemble_retriever.invoke(test_query)
print(f"\n:dart: Ensemble 검색 결과 ({len(ensemble_results)}개):")
print("=" * 60)
for i, doc in enumerate(ensemble_results[:2], 1):
    print(f"[{i}] {doc.page_content[:150]}...")

In [14]:
# ---------------------------------------------------
# MMR Retriever 생성 — 관련성·다양성 균형 검색
# ---------------------------------------------------
# ============================================================
# TODO: as_retriever(search_type="mmr")로 MMR 검색기를 만드세요
# 힌트: fetch_k=20 (후보), k=4 (최종), lambda_mult=0.5 (균형)
# 예상 결과: "MMR Retriever 생성 완료" 출력
# ============================================================

# MMR Retriever 생성
# fetch_k=20, k=4 → 20개 후보에서 다양성을 고려해 4개 선별

mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)


### 일반 검색 vs MMR 비교

In [15]:
# 넓은 주제 질문 — 유사 문서 중복이 발생하기 쉬운 질문으로 MMR 효과 부각
test_query = "2026년 달라지는 고용·복지 지원 제도를 정리해줘"

# 일반 유사도 검색
print(":mag: 일반 유사도 검색:")
print("=" * 60)
similarity_results = faiss_retriever.invoke(test_query)
for i, doc in enumerate(similarity_results, 1):
    print("=" * 60)
    print(f"[{i}] (p.{doc.metadata.get('page', '?')}) {doc.page_content[:200]}...")
    print("=" * 60)

# MMR 검색
print("\n:dart: MMR 검색 (다양성 고려):")
print("=" * 60)
mmr_results = mmr_retriever.invoke(test_query)
for i, doc in enumerate(mmr_results, 1):
    print("=" * 60)
    print(f"[{i}] (p.{doc.metadata.get('page', '?')}) {doc.page_content[:200]}...")
    print("=" * 60)


:mag: 일반 유사도 검색:
[1] (p.41) 1
2026년부터 이렇게 달라집니다
2026년부터
이렇게 달라집니다
금융·재정·조세01...
[2] (p.63) 23
2026년부터 이렇게 달라집니다
지방이전 기업 세제지원 제도 개선
재정경제부
www.moef.go.kr
지역균형발전을 지원하고 제도를 합리화하기 위하여, 지방이전 기업 세제지원 제도의 적용대상, 
감면기간을 확대하고 감면한도 및 사후관리를 신설합니다.
‌ 	 (대상지역 및 감면기간)
‌ 	 (감면한도) 지방투자누계액 × 70%+지방근무상시근로자 수...
[3] (p.71) 31
2026년부터 이렇게 달라집니다
고향사랑기부금 세액공제 확대
재정경제부
www.moef.go.kr
10만원 초과 20만원 이하 고향사랑기부금에 대한 세액공제율을 15→40%로 상향합니다.
* (현행) (~10만원) 100/110, (~2천만원) 15%[특별재난지역 30%]
(개정안) (~10만원) 100/110, (~20만원) 40%<신설> (~2천만...

:dart: MMR 검색 (다양성 고려):
[1] (p.41) 1
2026년부터 이렇게 달라집니다
2026년부터
이렇게 달라집니다
금융·재정·조세01...
[2] (p.94) 54
원자력안전관리부담금 산정기준 개편
원자력안전위원회 
www.nssc.go.kr
매년 업무량에 따라 변동되던 원자력안전관리부담금(이하 ‘부담금’)의 산정기준을 정액제로 
개편하여 납부대상자의 예측 가능성을 제고하였습니다.
‌   주요 원자력이용시설(한국수력원자력, 한국원자력연구원 등)은 운영 단계와 용량을 
고려한 정액제로 변경하고, 표준설계인가, ...
[3] (p.61) 21
2026년부터 이렇게 달라집니다
부분복귀하는 해외진출기업에 대한 소득세·법인세·관세 
감면 확대
재정경제부
www.moef.go.kr
유턴기업 지원 확대를 위하여, 국내사업장 신·증설 후 4년 이내에 국외사업장 축소를 완료하는 
기업도 유턴기업 세제지원 대상에 추가됩니다.
* (현)감면율 : (소득·법인세)

## 3. Ensemble + MMR 결합

최고의 성능을 위해 Ensemble Retriever와 MMR을 함께 사용할 수 있습니다.

In [17]:
# ---------------------------------------------------
# Ensemble + MMR 결합 — 최고 성능 검색기 구성
# ---------------------------------------------------
# ============================================================
# TODO: MMR을 적용한 FAISS Retriever와 BM25를 EnsembleRetriever로 결합하세요
# 힌트: weights=[0.4, 0.6]으로 의미 검색에 약간 더 가중치 부여
# 예상 결과: "Ensemble + MMR Retriever 생성 완료" 출력
# ============================================================

# 1단계: MMR을 적용한 FAISS Retriever
faiss_mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)

# 2단계: Ensemble: BM25(키워드) + FAISS(MMR) 결합
ensemble_mmr_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_mmr_retriever],
    weights=[.3, .6]
)

# 복합 질문 — 키워드 매칭 + 의미 검색 + 다양성이 모두 필요한 질문
test_query = "저소득 중증장애인을 위한 취업 프로그램과 수당은 어떤 것이 있나요?"

# Ensemble + MMR 검색
print(":dart: Ensemble + MMR 검색:")
print("=" * 60)
ensemble_results = ensemble_mmr_retriever.invoke(test_query)
for i, doc in enumerate(ensemble_results, 1):
    print("=" * 60)
    print(f"[{i}] (p.{doc.metadata.get('page', '?')}) {doc.page_content[:200]}...")
    print("=" * 60)

:dart: Ensemble + MMR 검색:
[1] (p.64) 24
장애인 표준사업장에 대한 세액감면 확대
재정경제부
www.moef.go.kr
장애인 일자리 창출 지원을 위하여, 장애인 표준사업장에 대한 소득세·법인세 감면율을 
확대합니다.
‌ 	 (종전) 3년 100% + 2년 50%
‌ 	 (개정) 3년 100% + 2년 50% + 5년 30%
개정내용은 2026년 1월 1일 이후 장애인 표준사업장으로 인증받...
[2] (p.66) 26
대학생 교육비 특별세액공제 소득요건 폐지
재정경제부
www.moef.go.kr
현재 시행 중인 본인과 부양가족의 교육비에 대한 15% 세액공제가 대학생 자녀의 아르바이트 
소득으로 인해 대상에서 제외되지 않도록 자녀의 소득요건*을 폐지합니다.
* 현재 자녀의 소득금액 합계액 100만원 초과 시 교육비 공제 불가
개정내용은 2026년 1월 1일 이후 지...
[3] (p.42) 2
금융·재정·조세
제 1 장
통합고용세액공제 공제액 구조 개편 및 사후관리 합리화
시행일: 2026년 1월 1일
자세한 내용은p.008
• 장기고용 유인강화 및 납세 협력 비용 경감을 위해 통합고용 세액공제의 공제액 구조를 개편하고 
사후관리를 합리화 합니다.
시행일: 2026년 1월 1일
웹툰콘텐츠 제작비용 세액공제 신설
시행일: 2026년 1월 1일...
[4] (p.92) · (주주총회 공시): 2026년 3월 1일(3월 1일 이후 시행되는 주주총회부터 적용)
	
· (임원보수·주식기준보상 공시): 2026년 5월 1일(2026년 반기보고서부터 적용)
금융위 누리집>보도자료>자본시장 접근성 제고를 위한 기업공시 개선 방안
금융위원회 공정시장과
☎ 02-2100-2688...
[5] (p.51) 11
2026년부터 이렇게 달라집니다
초등 저학년 예체능 학원비 세제지원 
재정경제부
www.moef.go.kr
현재 시행 중인 교육비 세액공제(15%) 대상에 2026년부터 초등학교 저학년 자녀의 예체능 
학원비도 포함됩니다.
‌ 	 

## 4. 고급 RAG 체인 구축

In [18]:
# ---------------------------------------------------
# 고급 RAG 체인 조립 — Ensemble+MMR 검색기 연결
# ---------------------------------------------------
# ============================================================
# TODO: ensemble_mmr_retriever, prompt, llm을 LCEL로 연결해 RAG 체인을 만드세요
# 힌트: {"context": ensemble_mmr_retriever, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
# 예상 결과: "고급 RAG 체인 생성 완료" 출력
# ============================================================

# 프롬프트
prompt = PromptTemplate.from_template(
    """당신은 문서 기반 질의응답을 수행하는 AI 어시스턴트입니다.
주어진 문맥을 바탕으로 질문에 답변해 주세요.

#Context:
{context}

#Question:
{question}

#Answer:"""
)

# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 고급 RAG 체인 — Ensemble+MMR 검색기 활용
advanced_rag_chain = (
    {"context": ensemble_mmr_retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(":white_check_mark: 고급 RAG 체인 생성 완료")

:white_check_mark: 고급 RAG 체인 생성 완료


### 성능 비교: 기본 vs 고급 RAG

> **실무 팁**: 두 답변을 비교할 때 단순히 길이나 어투가 아닌 **정보의 다양성**을 확인하세요. 고급 RAG 답변은 더 다양한 관점과 근거를 포함해야 합니다. 실제 프로젝트에서는 RAGAS 같은 RAG 평가 프레임워크로 정량적으로 비교합니다.

In [ ]:
# ---------------------------------------------------
# 기본 RAG vs 고급 RAG 답변 품질 비교
# ---------------------------------------------------
# ============================================================
# TODO: 동일한 질문을 basic_rag_chain과 advanced_rag_chain에 각각 invoke하세요
# 힌트: rag_chain.invoke(test_question)으로 두 체인의 답변을 비교
# 예상 결과: 고급 RAG가 더 다양하고 풍부한 답변을 생성함
# ============================================================

# 기본 RAG 체인 (비교용)


## 💡 핵심 정리

### Ensemble Retriever

**장점**:
- ✅ 키워드와 의미 검색의 장점 결합
- ✅ 정확한 용어와 유사 개념 모두 검색
- ✅ 검색 정확도 향상

**사용 시나리오**:
- 전문 용어가 많은 문서 (의료, 법률, 기술)
- 고유명사 검색이 중요한 경우

### MMR (Maximum Marginal Relevance)

**장점**:
- ✅ 검색 결과의 다양성 확보
- ✅ 중복 내용 제거
- ✅ 다양한 관점의 정보 제공

**사용 시나리오**:
- 포괄적인 정보가 필요한 경우
- 여러 관점의 답변이 필요한 경우

### 파라미터 튜닝 가이드

**Ensemble 가중치**:
```python
weights=[0.7, 0.3]  # 키워드 중심 (전문 용어 많을 때)
weights=[0.5, 0.5]  # 균형 (일반적)
weights=[0.3, 0.7]  # 의미 중심 (동의어 많을 때)
```

**MMR lambda_mult**:
```python
lambda_mult=1.0  # 관련성만 고려 (일반 검색과 동일)
lambda_mult=0.7  # 관련성 우선
lambda_mult=0.5  # 균형 (권장)
lambda_mult=0.3  # 다양성 우선
```

### 성능 향상 효과

| 기법 | 검색 정확도 | 다양성 | 처리 시간 |
|------|-----------|--------|----------|
| 기본 RAG | ⭐⭐⭐ | ⭐⭐ | 빠름 |
| + Ensemble | ⭐⭐⭐⭐ | ⭐⭐ | 보통 |
| + MMR | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | 약간 느림 |
| Ensemble + MMR | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | 보통 |

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 구성 | BM25 (Sparse) + FAISS with MMR (Dense) → EnsembleRetriever |
| MMR의 역할 | 유사도가 높지만 중복되지 않는 다양한 문서 선택 |
| BM25의 역할 | 키워드 정확 매칭으로 의미 검색이 놓치는 문서 보완 |
| Ensemble 가중치 | `[0.5, 0.5]` 기본 — 키워드 중심 도메인은 BM25 가중치를 높여요 |
| 응용 효과 | 기본 RAG 대비 동일 쿼리에서 더 다양하고 보완적인 문서 검색 |

### 고급 RAG 구성 전략

| 전략 | 구성 | 적합한 상황 |
|------|------|-------------|
| 기본 | FAISS Similarity | 단순 의미 검색으로 충분할 때 |
| 다양성 | FAISS MMR | 비슷한 문서 중복을 피하고 싶을 때 |
| 키워드 보완 | BM25 + FAISS Ensemble | 전문 용어, 고유명사가 많을 때 |
| **고급 (이 노트북)** | **BM25 + FAISS(MMR) Ensemble** | **다양성과 키워드 보완 모두 필요할 때** |

### 다음 노트북 예고

**03-Conversation-RAG** — 이전 대화 내용을 기억하면서 후속 질문을 처리하는 대화형 RAG를 배워요. `RunnableWithMessageHistory`와 세션 관리로 멀티턴 QA 시스템을 구축해요.